# Saudi Tweet Summary - "SAUDI IS BURNING"
What I found about this one tweet:
1. 4 notes were written on it: 2 by humans and 2 by AI note writers. Only Note 4 (human) was shown on X.
2. Notes 2 and 3 are AI. Their writers have the status `apiEarnedIn`, and they are the 2 most active AI accounts on X.
3. People rated all 4 notes about the same overall, but two groups of raters disagreed a lot.
4. Note 4 was not shown because it had more votes or a higher % Helpful. The algorithm looks at *which* raters agreed, and that part is not in the data.

## The map
![Saudi tweet map](diagrams/saudi_tweet.svg)

Editable version: `diagrams/saudi_tweet.drawio` (open it at app.diagrams.net). The arrows show how the files connect.

In [1]:
import glob

import pandas as pd

pd.set_option("display.max_colwidth", 80)

# ids are 19 digits, read them as strings so pandas doesn't round them
ids = {"noteId": str, "tweetId": str, "noteAuthorParticipantId": str, "raterParticipantId": str, "participantId": str}

folder = "dataset/tweet_2109031558967615703/"
combined = pd.read_csv(folder + "combined.csv", dtype=ids)
notes = pd.read_csv(folder + "notes.csv", dtype=ids)
status = pd.read_csv(folder + "note_status.csv", dtype=ids)
ratings = pd.read_csv(folder + "ratings.csv", dtype=ids)
requests = pd.read_csv(folder + "note_requests.csv", dtype=ids)

tweet_time = pd.to_datetime(combined["tweet_created_at"][0], utc=True)

print("tweet:", combined["tweet_text"][0])
print("posted:", tweet_time)
print("notes:", len(notes), "| ratings:", len(ratings), "| raters:", ratings["raterParticipantId"].nunique())

tweet: SAUDI IS BURNING
posted: 2026-10-10 21:21:22+00:00
notes: 4 | ratings: 456 | raters: 263


## 1. The 4 notes

In [2]:
# minutes between the tweet and each note
notes["written"] = pd.to_datetime(notes["createdAtMillis"], unit="ms", utc=True)
notes["minutes_after_tweet"] = (notes["written"] - tweet_time).dt.total_seconds() / 60

# was the note shown on X?
status["shown"] = status["currentStatus"] == "CURRENTLY_RATED_HELPFUL"
notes = notes.merge(status[["noteId", "shown"]], on="noteId")

notes[["note_label", "writer_type", "minutes_after_tweet", "shown", "summary"]]

,note_label,writer_type,minutes_after_tweet,shown,summary
0,Note 1,Human,15.600250,False,Mario claims saudi arabia is burning after an attack on kkia this is poor j...
1,Note 2,AI,48.454100,False,Saudi Arabia faces targeted Houthi attacks on Riyadh airport causing localiz...
2,Note 3,AI,118.152033,False,Saudi Arabia is not &quot;burning.&quot; Official reports detail a specific ...
3,Note 4,Human,119.593150,True,“Saudi is burning” misleadingly implies nationwide fires. Saudi authoritie...


All 4 notes say the same thing: only the airport was hit, the country is not "burning".
Note 1 came 16 minutes after the tweet. Note 4 was written last (2 hours after the tweet), but it is the only one that was shown.

## 2. Which notes are AI, and who are the AI writers?

X has an API for AI note writers (since July 2025). In X's user file these accounts have the status `apiEarnedIn`.
I added the writer's status to `notes.csv` as `writer_enrollmentState`.
- **Status** = `enrollmentState` in X's user file (`data/userEnrollment-00000.tsv`). It is not in the original ratings or notes files,
  so I copied it in: `writer_enrollmentState` in `notes.csv` and `rater_enrollmentState` in `ratings.csv`.
- **How many notes an account wrote** comes from X's notes files (`data/notes/notes-*.tsv`), by counting rows per `noteAuthorParticipantId`.

In [3]:
notes[["note_label", "writer_type", "writer_enrollmentState", "noteAuthorParticipantId"]]

,note_label,writer_type,writer_enrollmentState,noteAuthorParticipantId
0,Note 1,Human,earnedIn,F00B4604B909134122258F4D4E7A5FA933D357089AAFC143EA78CBA815DB400C
1,Note 2,AI,apiEarnedIn,F14ECB41C233875132EBFAC0D72206681017FDA097BDE992E887E10E2EB398A7
2,Note 3,AI,apiEarnedIn,A9CD635B3039333BE6A70CDA85EAFE27FA13CFCF9031819B797887C80BFA3366
3,Note 4,Human,earnedIn,38803E6524398B188C6CA75065532A23784FC2E708FD66FB7F177BF79725E2B8


In [4]:
# AI writers get tweets from X's request feed. When did this tweet get into it?
request = requests.iloc[0]
ai_feed_time = pd.to_datetime(int(request["apiXlFeedEligibleAtMillis"]), unit="ms", utc=True)

print("tweet got into the AI writers' feed after", round((ai_feed_time - tweet_time).total_seconds() / 60), "minutes")
print("human request feed:", request["noteRequestFeedEligibleAtMillis"], "(-1 means never)")

tweet got into the AI writers' feed after 32 minutes
human request feed: -1 (-1 means never)


To check how active these AI writers are, I count the notes of every AI account in all of X's data.
This reads all notes ever written, so it takes about a minute.

In [5]:
# all notes ever written (5 files), only 2 columns
parts = []
for file in sorted(glob.glob("data/notes/notes-*.tsv")):
    part = pd.read_csv(file, sep="\t", usecols=["noteAuthorParticipantId", "createdAtMillis"], dtype=ids)
    parts.append(part)
all_notes = pd.concat(parts)

# X's user file, to get each writer's status
users = pd.read_csv("data/userEnrollment-00000.tsv", sep="\t", usecols=["participantId", "enrollmentState"], dtype=ids)
users = users.set_index("participantId")
all_notes["status"] = all_notes["noteAuthorParticipantId"].map(users["enrollmentState"])

# number of notes for each AI account, most active first
ai_notes = all_notes[all_notes["status"] == "apiEarnedIn"]
notes_per_ai = ai_notes["noteAuthorParticipantId"].value_counts()

print("AI accounts that wrote notes:", len(notes_per_ai))
notes_per_ai.head(5)

AI accounts that wrote notes: 36


noteAuthorParticipantId
A9CD635B3039333BE6A70CDA85EAFE27FA13CFCF9031819B797887C80BFA3366    46308
F14ECB41C233875132EBFAC0D72206681017FDA097BDE992E887E10E2EB398A7    45737
A0A7C54DAD6E5E83B3862F3B499C5019EBA29A8A5142765275BCCC5BDEC32215    45306
B3E2C89FD19296525A8117746AFB6E96AEF503E4EB9B1C1DF5E112D2E2D0BE3C    45211
D6D5E740F63DC34D81BD735B2AFCB39D253A6AC476710B31032419C2E47BD85B    44973
Name: count, dtype: int64

In [6]:
# where do our 4 writers fall?
ai_ranking = list(notes_per_ai.index)

for _, note in notes.iterrows():
    writer = note["noteAuthorParticipantId"]
    total = (all_notes["noteAuthorParticipantId"] == writer).sum()
    if writer in ai_ranking:
        rank = ai_ranking.index(writer) + 1
        print(note["note_label"], "- AI,   ", total, "notes ever, rank", rank, "of", len(ai_ranking), "AI accounts")
    else:
        print(note["note_label"], "- human,", total, "notes ever")

Note 1 - human, 3 notes ever
Note 2 - AI,    45737 notes ever, rank 2 of 36 AI accounts
Note 3 - AI,    46308 notes ever, rank 1 of 36 AI accounts


Note 4 - human, 1 notes ever


In [7]:
# do the AI writers rate notes? (people have to rate before they can write, AI writers don't)
ai_writers = notes.loc[notes["writer_type"] == "AI", "noteAuthorParticipantId"]
print("ratings on this tweet given by the 2 AI writers:", ratings["raterParticipantId"].isin(ai_writers).sum())

ratings on this tweet given by the 2 AI writers: 0


- Notes 2 and 3 are AI: their writers have the status `apiEarnedIn`.
- Out of all AI accounts on X, the writer of Note 3 has written the most notes (rank 1) and the writer of Note 2 is rank 2.
  Both have written about 46,000 notes since late 2025, around 150 a day. No person writes like that.
  The top 5 AI accounts are all close (about 45,000 notes each), so these two are part of a small group of very active AI writers.
- The tweet got into the AI writers' feed after 32 minutes, and the first AI note (Note 2) came 16 minutes later.
  It never got into the human request feed.
- Note 4 was written by a human, and it was their first note ever.
- The AI accounts are anonymous IDs, so I can't tell which company or model is behind them.

## 3. How people rated each note, and why
456 ratings is for **all 4 notes together** (one row in `ratings.csv` = one person's rating of one note).

In [8]:
# count the ratings of each note
for label in ["Note 1", "Note 2", "Note 3", "Note 4"]:
    levels = ratings.loc[ratings["note_label"] == label, "helpfulnessLevel"]
    helpful = (levels == "HELPFUL").sum()
    not_helpful = (levels == "NOT_HELPFUL").sum()
    print(label, "-", len(levels), "ratings:", helpful, "Helpful,", not_helpful, "Not Helpful,",
          round(helpful / len(levels) * 100), "% Helpful")

Note 1 - 113 ratings: 87 Helpful, 25 Not Helpful, 77 % Helpful
Note 2 - 101 ratings: 77 Helpful, 21 Not Helpful, 76 % Helpful
Note 3 - 77 ratings: 64 Helpful, 12 Not Helpful, 83 % Helpful
Note 4 - 165 ratings: 135 Helpful, 27 Not Helpful, 82 % Helpful


In [9]:
# why people said Not Helpful (% of each note's Not Helpful ratings)
not_helpful = ratings[ratings["helpfulnessLevel"] == "NOT_HELPFUL"]
reasons = ["notHelpfulNoteNotNeeded", "notHelpfulMissingKeyPoints", "notHelpfulArgumentativeOrBiased", "notHelpfulIncorrect"]

(not_helpful.groupby("note_label")[reasons].mean() * 100).round()

,notHelpfulNoteNotNeeded,notHelpfulMissingKeyPoints,notHelpfulArgumentativeOrBiased,notHelpfulIncorrect
note_label,,,,
Note 1,44.0,24.0,40.0,28.0
Note 2,48.0,43.0,24.0,19.0
Note 3,67.0,42.0,33.0,17.0
Note 4,41.0,41.0,41.0,33.0


In [10]:
# 10 example ratings (1 = the rater ticked that reason)
columns = ["note_label", "rater_modelingGroup", "helpfulnessLevel",
           "helpfulGoodSources", "helpfulClear", "notHelpfulNoteNotNeeded", "notHelpfulArgumentativeOrBiased"]
ratings[columns].sample(10, random_state=1)

,note_label,rater_modelingGroup,helpfulnessLevel,helpfulGoodSources,helpfulClear,notHelpfulNoteNotNeeded,notHelpfulArgumentativeOrBiased
67,Note 1,24.0,HELPFUL,1,1,0,0
146,Note 2,24.0,HELPFUL,1,1,0,0
322,Note 4,19.0,NOT_HELPFUL,0,0,0,0
250,Note 3,24.0,HELPFUL,1,1,0,0
223,Note 3,24.0,HELPFUL,1,0,0,0
366,Note 4,29.0,NOT_HELPFUL,0,0,1,1
380,Note 4,24.0,HELPFUL,0,0,0,0
427,Note 4,24.0,HELPFUL,1,1,0,0
436,Note 4,24.0,HELPFUL,1,1,0,0
92,Note 1,24.0,HELPFUL,1,0,0,0


- All 4 notes got about the same share of Helpful ratings (76-83%). Note 3 had the highest share, but it was not shown.
- For most notes, the top Not Helpful reason is "note not needed": those people don't think the tweet is misleading.
- Note 1 got a lot of "argumentative or biased" and "incorrect". It calls the post "poor journalism" and says there were no deaths.

## 4. Who the raters are

I added each rater's info from X's user file to `ratings.csv` (the `rater_...` columns).
Groups = groups of raters that X makes based on region, country or language (X's guide, "Note ranking"). The numbers (24, 13) are just IDs, X doesn't say which region each one is.

In [11]:
raters = ratings.drop_duplicates("raterParticipantId")

print("raters:", len(raters))
print("status:", raters["rater_enrollmentState"].value_counts().to_dict())
print("population:", raters["rater_modelingPopulation"].value_counts().to_dict())
print("biggest groups:", raters["rater_modelingGroup"].value_counts().head(3).to_dict())

raters: 263
status: {'earnedIn': 173, 'newUser': 76, 'earnedOutAcknowledged': 10, 'atRisk': 2, 'removed': 2}
population: {'EXPANSION': 204, 'CORE': 57, 'EXPANSION_PLUS': 2}
biggest groups: {24.0: 184, 13.0: 37, 27.0: 9}


In [12]:
# % Helpful from group 24 and group 13, for each note
for label in ["Note 1", "Note 2", "Note 3", "Note 4"]:
    note_ratings = ratings[ratings["note_label"] == label]
    group_24 = note_ratings[note_ratings["rater_modelingGroup"] == 24]
    group_13 = note_ratings[note_ratings["rater_modelingGroup"] == 13]
    print(label,
          "| group 24:", len(group_24), "ratings,", round((group_24["helpfulnessLevel"] == "HELPFUL").mean() * 100), "% Helpful",
          "| group 13:", len(group_13), "ratings,", round((group_13["helpfulnessLevel"] == "HELPFUL").mean() * 100), "% Helpful")

Note 1 | group 24: 83 ratings, 92 % Helpful | group 13: 17 ratings, 29 % Helpful
Note 2 | group 24: 70 ratings, 91 % Helpful | group 13: 15 ratings, 53 % Helpful
Note 3 | group 24: 58 ratings, 97 % Helpful | group 13: 9 ratings, 44 % Helpful
Note 4 | group 24: 119 ratings, 96 % Helpful | group 13: 22 ratings, 45 % Helpful


In [13]:
# how many groups, and which population each of the 2 big groups belongs to
print("rater groups among these raters:", raters["rater_modelingGroup"].nunique())
print("ratings by population:", ratings["rater_modelingPopulation"].value_counts().to_dict())
for group in [24, 13]:
    people = raters[raters["rater_modelingGroup"] == group]
    print("group", group, "-", len(people), "raters, population:", people["rater_modelingPopulation"].value_counts().to_dict())

rater groups among these raters: 17
ratings by population: {'EXPANSION': 356, 'CORE': 98, 'EXPANSION_PLUS': 2}
group 24 - 184 raters, population: {'EXPANSION': 184}
group 13 - 37 raters, population: {'CORE': 37}


In [14]:
# how many of the 4 notes did each person rate?
notes_rated = ratings["raterParticipantId"].value_counts()
print("people who rated 1, 2, 3 or all 4 notes:")
print(notes_rated.value_counts().sort_index().to_string())

people who rated 1, 2, 3 or all 4 notes:
count
1    162
2     44
3     22
4     35


Most raters (204 of 263) are in the EXPANSION population, and 184 are in group 24.
Group 24 rated every note Helpful (91-97%), but group 13 was split (29-53%). So the two groups didn't agree on any of the notes.

## 5. How the approval works, and why Note 4 was shown

From X's documentation (communitynotes.x.com, "Note ranking"): the algorithm learns from everyone's past ratings which raters
usually disagree with each other. Then it gives each note a score for how helpful it is to raters on **both** sides.
A note is shown only if that score is high enough, not just because many people liked it.
X runs a few versions of the model (Core, Expansion, ...), and `note_status.csv` says which one made the decision.

**Core model vs Expansion model** (from X's guide, and checked in the data below):
- X puts every rater in a population (`rater_modelingPopulation`) by **where they are**:
  **CORE** = raters in "established areas" (countries where Community Notes has existed for a long time),
  **EXPANSION** = raters in "areas where Community Notes has recently launched", **EXPANSION_PLUS** = a newer, wider extension.
- The **Core model** only scores notes where most ratings come from Core areas, using Core raters.
  The **Expansion model** runs the same algorithm on all ratings, from Core and Expansion areas. If both score a note, Core wins.
- `note_status.csv` says which one decided: `currentDecidedBy`. Here `currentCoreStatus` is empty for all 4 notes
  (the Core model gave no result), so the decision came from the Expansion model.

In [15]:
status[["note_label", "currentStatus", "currentCoreStatus", "currentExpansionStatus", "currentDecidedBy"]]

,note_label,currentStatus,currentCoreStatus,currentExpansionStatus,currentDecidedBy
0,Note 1,NEEDS_MORE_RATINGS,NaN,FIRM_REJECT,ExpansionModel (v1.1)
1,Note 2,NEEDS_MORE_RATINGS,NaN,FIRM_REJECT,ExpansionModel (v1.1)
2,Note 3,NEEDS_MORE_RATINGS,NaN,NEEDS_MORE_RATINGS,ExpansionModel (v1.1)
3,Note 4,CURRENTLY_RATED_HELPFUL,NaN,CURRENTLY_RATED_HELPFUL,ExpansionModel (v1.1)


In [16]:
# the moment Note 4 was shown
shown_at = int(status.loc[status["note_label"] == "Note 4", "timestampMillisOfFirstNonNMRStatus"].iloc[0])
hours = (pd.to_datetime(shown_at, unit="ms", utc=True) - tweet_time).total_seconds() / 3600
print("Note 4 was shown", round(hours, 1), "hours after the tweet")

# what every note looked like at that moment
before = ratings[ratings["createdAtMillis"] <= shown_at]
for label in ["Note 1", "Note 2", "Note 3", "Note 4"]:
    note_ratings = before[before["note_label"] == label]
    core = note_ratings[note_ratings["rater_modelingPopulation"] == "CORE"]
    print(label,
          "| ratings so far:", len(note_ratings),
          "| % Helpful:", round((note_ratings["helpfulnessLevel"] == "HELPFUL").mean() * 100),
          "| CORE % Helpful:", round((core["helpfulnessLevel"] == "HELPFUL").mean() * 100))

Note 4 was shown 7.5 hours after the tweet
Note 1 | ratings so far: 99 | % Helpful: 74 | CORE % Helpful: 33
Note 2 | ratings so far: 92 | % Helpful: 74 | CORE % Helpful: 46
Note 3 | ratings so far: 57 | % Helpful: 79 | CORE % Helpful: 38
Note 4 | ratings so far: 86 | % Helpful: 73 | CORE % Helpful: 36


- Most ratings on this tweet came from Expansion raters (356 of 456), so it is not a Core-model note:
  `currentCoreStatus` is empty for all 4 notes, and the **Expansion model** decided.
- **Note 4:** shown, 7.5 hours after the tweet. **Note 3:** `NEEDS_MORE_RATINGS` (it had the fewest ratings).
  **Notes 1 and 2:** `FIRM_REJECT`, they will not be shown.
- When Note 4 was shown, it had **fewer ratings** than Notes 1 and 2 (86 vs 99 and 92) and about the **same % Helpful**.
  CORE raters were split on every note (33-46% Helpful).
- So the numbers I can see don't explain why Note 4 won. The algorithm uses each rater's viewpoint, learned from all their
  past ratings, and that is not in X's download.
- Next step: run X's open-source algorithm (github.com/twitter/communitynotes) on the ratings, to get each rater's and each note's score.

## 6. Do all ratings count?

From X's documentation ("Contributor helpfulness scores"): not every rating is used in the final round of scoring.
A rater's ratings only count if they have **at least 10 ratings**, and a **rater helpfulness score of at least 0.66**
(about 2 out of 3 of their past ratings matched the final result). Raters with a bad track record are left out.

I can check the first rule with the data: how many ratings each Saudi rater had made before Note 4 was shown.
This reads all 9 ratings files (238 million ratings), so it takes about 2 minutes.

In [17]:
# count every rating our 263 raters made before Note 4 was shown
saudi_raters = set(ratings["raterParticipantId"])
counts = pd.Series(0, index=sorted(saudi_raters))

for file in sorted(glob.glob("data/ratings/ratings-*.tsv")):
    for chunk in pd.read_csv(file, sep="\t", usecols=["raterParticipantId", "createdAtMillis"], dtype=ids, chunksize=5_000_000):
        mine = chunk[chunk["raterParticipantId"].isin(saudi_raters) & (chunk["createdAtMillis"] <= shown_at)]
        counts = counts.add(mine["raterParticipantId"].value_counts(), fill_value=0)

print("median ratings per rater before Note 4 was shown:", counts.median())
print("raters with fewer than 10 ratings:", (counts < 10).sum(), "of", len(counts))

median ratings per rater before Note 4 was shown: 87.0
raters with fewer than 10 ratings: 21 of 263


In [18]:
# each note at that moment, keeping only raters with 10+ ratings
before = ratings[ratings["createdAtMillis"] <= shown_at].copy()
before["rater_count"] = before["raterParticipantId"].map(counts)
kept = before[before["rater_count"] >= 10]

for label in ["Note 1", "Note 2", "Note 3", "Note 4"]:
    all_ratings = before[before["note_label"] == label]
    kept_ratings = kept[kept["note_label"] == label]
    print(label,
          "| ratings so far:", len(all_ratings),
          "| from raters with 10+ ratings:", len(kept_ratings),
          "| % Helpful among them:", round((kept_ratings["helpfulnessLevel"] == "HELPFUL").mean() * 100))

Note 1 | ratings so far: 99 | from raters with 10+ ratings: 95 | % Helpful among them: 74
Note 2 | ratings so far: 92 | from raters with 10+ ratings: 87 | % Helpful among them: 74
Note 3 | ratings so far: 57 | from raters with 10+ ratings: 56 | % Helpful among them: 79
Note 4 | ratings so far: 86 | from raters with 10+ ratings: 79 | % Helpful among them: 73


- Most raters are experienced: the median rater had 87 ratings before this tweet. Only 21 of 263 had fewer than 10 (mostly new users).
- Removing them changes very little (2-8% of each note's ratings), and Note 4 is still not ahead in count or % Helpful.
- So the "10 ratings" rule doesn't explain why Note 4 won. What's left is each rater's **helpfulness score** and **viewpoint**
  in X's model. Both are not in the download, they come from running X's open-source algorithm. That is the next step.